## MCP Servers in LangChain Agents

In [1]:
%pip install -q langchain langchain-openai langchain-mcp-adapters python-dotenv pywin32

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install --force-reinstall --no-cache-dir pywin32 

# restart the kernel after running the above command to ensure that the 
# pywin32 module is properly registered and available for use in your Python environment.

   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ----------- ---------------------------- 2.1/7.0 MB 11.3 MB/s eta 0:00:01
   ------------------------- -------------- 4.5/7.0 MB 11.1 MB/s eta 0:00:01
   -------------------------------------- - 6.8/7.0 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 11.2 MB/s  0:00:00
  Attempting uninstall: pywin32
    Found existing installation: pywin32 312
    Uninstalling pywin32-312:
      Successfully uninstalled pywin32-312
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient  
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, AIMessage, ToolMessage

from dotenv import load_dotenv

In [20]:
load_dotenv(override=True)  # Load environment variables from .env file

True

In [21]:
model = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)


async def get_mcp_client():

    return MultiServerMCPClient(
        {
            "tapetide": {
                "transport": "http",  # HTTP-based remote server
                "url": "https://mcp.tapetide.com/mcp",
                "headers": {
                    "Authorization": f"Bearer {os.getenv('TAPETIDE_API_KEY')}"
                }
            },
            "tavily": {
                "transport": "http",
                "url": "https://mcp.tavily.com/mcp/",
                "headers": {
                    "X-Tavily-Access-Mode": "keyless"
                }
            }
        }
    )

async def get_agent():
    client = await get_mcp_client()
    tools = await client.get_tools()
    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt="""You are a helpful assistant. You can use the TapeTide tool to get the stock related information.
        Use the Tavily tools to perform general web search operations."""
    )
    return agent


In [22]:
agent = await get_agent()

In [26]:
response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what is the stock value for Hexaware?"}]}
)
print(f"Agent response : {response['messages'][-1].content}")

for message in response["messages"]:
    if type(message) is HumanMessage:
        print(f"User : {message.content}")
    if type(message) is AIMessage and message.tool_calls :
        print(f"Agent requested Tool Calls : {message.tool_calls}")
    if type(message) is AIMessage and not message.tool_calls:
        print(f"Agent : {message.content}")
    if type(message) is ToolMessage:
        print(f"Tool : {message.content}")

Agent response : Hexaware Technologies (**NSE: HEXT**) is trading at approximately **₹582.35 per share**, down **₹8.45 (1.43%)** from the previous close.

- **Market capitalization:** ₹35,582.52 crore  
- **52-week range:** ₹400.20–₹807.75  
- **P/E:** 26.46  
- **Quote timestamp:** 26 October 2026, 05:56 UTC
User : what is the stock value for Hexaware?
Agent requested Tool Calls : [{'name': 'search_stocks', 'args': {'query': 'Hexaware', 'limit': 5}, 'id': 'call_b4b3ueJPaPFb32EskDe4310d', 'type': 'tool_call'}]
Tool : [{'type': 'text', 'text': '{\n  "data": [\n    {\n      "symbol": "HEXT",\n      "name": "Hexaware Technologies Ltd",\n      "bse_code": "544362",\n      "nse_symbol": "HEXT",\n      "sector": "Information Technology",\n      "industry": "Computer Software & Consulting",\n      "macro_sector": "Information Technology",\n      "is_active": true,\n      "stock_type": "EQUITY"\n    }\n  ],\n  "meta": {\n    "total": 1,\n    "limit": 5,\n    "offset": 0\n  }\n}', 'id': 'lc_581

In [27]:
response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "What happened at the last wimbledon?"}]}
)
print(f"Agent response : {response['messages'][-1].content}")

for message in response["messages"]:
    if type(message) is HumanMessage:
        print(f"User : {message.content}")
    if type(message) is AIMessage and message.tool_calls :
        print(f"Agent requested Tool Calls : {message.tool_calls}")
    if type(message) is AIMessage and not message.tool_calls:
        print(f"Agent : {message.content}")
    if type(message) is ToolMessage:
        print(f"Tool : {message.content}")

Agent response : If you mean the **most recently completed Wimbledon I can reliably summarize—2025**:

- **Men’s singles:** Jannik Sinner defeated Carlos Alcaraz **4–6, 6–4, 6–4, 6–4**. It was Sinner’s first Wimbledon title, making him the first Italian men’s singles champion.
- **Women’s singles:** Iga Świątek beat Amanda Anisimova **6–0, 6–0** in just 57 minutes—one of the most dominant Grand Slam finals ever. It was Świątek’s first Wimbledon title and made her the first Polish singles champion there.
- The men’s final was especially notable because it was a rematch of the French Open final, where Alcaraz had beaten Sinner.

Since it’s now October 2026, **Wimbledon 2026 would technically be the last tournament**; I’m unable to verify its results right now.
User : What happened at the last wimbledon?
Agent requested Tool Calls : [{'name': 'tavily_search', 'args': {'query': "2025 Wimbledon Championships results men's women's singles finals highlights controversies", 'topic': 'general',